# FrameSat-AI: Kaggle Experiment Orchestrator
This notebook coordinates the environment validation, path discovery, configuration generation, training execution, post-training evaluation, and output packaging for FrameSat-AI. It is built to be deterministic, idempotent, and highly debuggable.

### Step 1: Bootstrap Orchestrator
Discover the project workspace path to import the shared `KaggleOrchestrator` helper.

In [ ]:
import os
import sys

# Dynamic discovery of bundle path
bundle_path = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "training" in dirs and "evaluation" in dirs:
        bundle_path = root
        break
if not bundle_path:
    if os.path.exists("/kaggle/working/training/train.py"):
        bundle_path = "/kaggle/working"

if not bundle_path:
    raise RuntimeError("Could not find FrameSat-AI source code bundle.")

if bundle_path not in sys.path:
    sys.path.insert(0, bundle_path)

from framesat_kaggle.kaggle_orchestrator import KaggleOrchestrator
orchestrator = KaggleOrchestrator()
orchestrator.bundle_path = bundle_path

print(f"\033[32m✔\033[0m Bootstrapped and loaded KaggleOrchestrator from: {bundle_path}")

### Step 2: Setup Environment
Validate CUDA availability and identify platform specifications.

In [ ]:
orchestrator.setup_environment()

### Step 3: Discover Resources
Scan input directories for datasets, metadata databases, and model checkpoints.

In [ ]:
orchestrator.discover_resources()

### Step 4: Run Pre-Flight Validations
Assert RIFE weight directory integrity, scene counts, and available disk space.

In [ ]:
orchestrator.validate_environment()

### Step 5: Resolve Configuration & Runtime Manifest
Generate overridden configurations, parse the resolved RIFE version, check for auto-resume paths, and save `runtime_manifest.json` metadata.

In [ ]:
orchestrator.resolve_runtime_config()

### Step 6: Launch Training
Invoke `train.py` using the resolved configuration, streaming logs in real time.

In [ ]:
orchestrator.launch_training()

### Step 7: Post-Training Scientific Evaluation
Evaluate the best model checkpoint against baseline metrics.

In [ ]:
orchestrator.evaluate()

### Step 8: Package & Export Outputs
Restructure training assets, compress into `experiment_001.tar.gz`, and print the final experiment summary.

In [ ]:
orchestrator.package_outputs()